# Imports

In [1]:
from ultralytics import YOLO
from pathlib import Path
import torch
import pandas as pd

if torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

# Configuration

In [2]:
DATA_YAML = "CS2 Dataset/YOLO_COCO/data.yaml"

MODEL_WEIGHTS = "Models/yolov8n.pt"

# Output structure
INDIV_RESULTS_ROOT = Path("Individual Results")
GROUP_RESULTS_ROOT = Path("Group Results")
MODEL_NAME = "YOLOv8_SignDetector"

TRAIN_NAME = f"{MODEL_NAME}_Train"
TEST_NAME  = f"{MODEL_NAME}_Test"

TRAIN_DIR = INDIV_RESULTS_ROOT / TRAIN_NAME
TEST_DIR  = INDIV_RESULTS_ROOT / TEST_NAME

# Training params
EPOCHS = 100
IMG_SIZE = 640
BATCH = 8
WORKERS = 2


# Train

In [3]:
model = YOLO(MODEL_WEIGHTS)

results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    workers=WORKERS,
    device=device,         
    project=str(INDIV_RESULTS_ROOT),
    name=TRAIN_NAME,
    patience=20,
    cache=False,
    exist_ok=True,

    # --- AUGMENTATION ---
    degrees=10.0,        # small rotations (realistic for street photos)
    translate=0.1,       # camera shift
    scale=0.5,           # distance variation
    shear=0.0,           # avoid sign distortion
    perspective=0.0,     # avoid unrealistic warping

    hsv_h=0.015,         # lighting variation
    hsv_s=0.6,
    hsv_v=0.4,

    fliplr=0.5,          # left/right OK for signs
    flipud=0.0,          # never flip vertically

    mosaic=0.0,          # ❗ disable (small objects)
    mixup=0.0            # ❗ disable (hurts localisation)
)

New https://pypi.org/project/ultralytics/8.3.250 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.248 🚀 Python-3.11.9 torch-2.9.1 MPS (Apple M4)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=CS2 Dataset/YOLO_COCO/data.yaml, degrees=10.0, deterministic=True, device=mps, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.6, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=Models/yolov8n.pt, momentum=0.937, mosaic=0.0, multi_scale=False, name=YOLOv8_SignDetector_Train, nbs=64, nms=False, o

# Test

In [4]:
BEST_MODEL = TRAIN_DIR / "weights" / "best.pt"
model = YOLO(BEST_MODEL)


metrics = model.val(
    data=DATA_YAML,
    split="test",
    project=str(INDIV_RESULTS_ROOT),
    name=TEST_NAME,
)


results_dict = {
    "Model Name": MODEL_NAME,
    "mAP@0.5": metrics.box.map50,
    "mAP@0.5:0.95": metrics.box.map,
    "Precision": metrics.box.mp,
    "Recall": metrics.box.mr,
    "Inference speed (ms)": metrics.speed["inference"]
}

df = pd.DataFrame([results_dict])
csv_path = GROUP_RESULTS_ROOT / f"{MODEL_NAME}.csv"
df.to_csv(csv_path, index=False)
display(df)

Ultralytics 8.3.248 🚀 Python-3.11.9 torch-2.9.1 CPU (Apple M4)
Model summary (fused): 72 layers, 3,006,818 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.1±0.3 ms, read: 1411.4±739.3 MB/s, size: 3861.3 KB)
val: Scanning /Users/matt/Documents/GitHub/Advanced-Computer-Vision/CS2 Dataset/YOLO_COCO/labels/test.cache... 97 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 97/97 346.5Kit/s 0.0s
val: /Users/matt/Documents/GitHub/Advanced-Computer-Vision/CS2 Dataset/YOLO_COCO/images/test/10d1875e-20251230_121901.jpg: corrupt JPEG restored and saved
val: /Users/matt/Documents/GitHub/Advanced-Computer-Vision/CS2 Dataset/YOLO_COCO/images/test/111da332-20251223_152248.jpg: corrupt JPEG restored and saved
val: /Users/matt/Documents/GitHub/Advanced-Computer-Vision/CS2 Dataset/YOLO_COCO/images/test/1d3d6802-20251230_122701.jpg: corrupt JPEG restored and saved
val: /Users/matt/Documents/GitHub/Advanced-Computer-Vision/CS2 Dataset/YOLO_COCO/images/test/1dbb2647-20251223_151149.

,Model Name,mAP@0.5,mAP@0.5:0.95,Precision,Recall,Inference speed (ms)
0,YOLOv8_SignDetector,0.722517,0.576189,0.849977,0.643484,76.403208
